In [ ]:
sudo kubeadm reset -f
sudo rm -rf /var/lib/etcd
sudo rm -rf /etc/kubernetes/

systemctl stop kubelet containerd
systemctl disable kubelet containerd

pkill -9 kubelet
pkill -9 containerd
pkill -9 containerd-shim
pkill -9 runc

umount -R /var/lib/kubelet 2>/dev/null
umount -R /run/containerd 2>/dev/null

mount | grep kubelet
mount | grep containerd

cat /proc/*/cgroup | grep kubepods 2>/dev/null

systemctl stop containerd
systemctl kill containerd

rm -rf /run/containerd
rm -rf /var/lib/containerd
rm -rf /var/lib/kubelet
rm -rf /etc/kubernetes
rm -rf /etc/cni/net.d
rm -rf /opt/cni/bin/*

ip link delete cni0 2>/dev/null
ip link delete flannel.1 2>/dev/null
ip link delete docker0 2>/dev/null
ip link delete vxlan.calico

Remove fennal

In [ ]:
kubectl delete -f https://github.com/flannel-io/flannel/releases/latest/download/kube-flannel.yml

sudo rm -rf /etc/cni/net.d/*flannel
sudo ip link delete cni0 2>/dev/null || true
sudo ip link delete flannel.1 2>/dev/null || true
sudo rm -rf /var/lib/cni/flannel

kubectl delete configmap -n kube-system kube-flannel-cfg 2>/dev/null || true

Install Calico

In [ ]:
kubectl create -f https://raw.githubusercontent.com/projectcalico/calico/v3.29.4/manifests/tigera-operator.yaml

wget https://raw.githubusercontent.com/projectcalico/calico/v3.29.4/manifests/custom-resources.yaml


In [ ]:
# Edit the custom-resources.yaml file
vi custom-resources.yaml

In [ ]:
apiVersion: operator.tigera.io/v1
kind: Installation
metadata:
  name: default
spec:
  calicoNetwork:
    ipPools:
    - name: default-ipv4-ippool
      blockSize: 26
      cidr: 10.244.0.0/16  # Change this to your pod CIDR
      encapsulation: VXLANCrossSubnet  # Use VXLANCrossSubnet for cross-subnet traffic
      natOutgoing: Enabled
      nodeSelector: all()

In [ ]:
# Create the custom resources
kubectl create -f custom-resources.yaml

In [ ]:
kubectl get pods -n calico-system -w

---

In [ ]:
kubectl get pods --all-namespaces -o wide

kubectl get svc --all-namespaces -o wide

curl -H "Host: headlamp.voip.local" http://172.16.6.90

---

In [ ]:
# Remove ingress
helm uninstall ingress-nginx -n ingress-nginx

# Remove Headlamp temporarily
helm uninstall my-headlamp -n headlamp

# Remove kube-vip DaemonSet on workers (keep master static pods)
kubectl delete ds kube-vip-worker-ds -n kube-system

# (Optional but clean) Remove Calico and reinstall fresh later
# kubectl delete -f custom-resources.yaml
# kubectl delete -f tigera-operator.yaml

In [ ]:
kubectl create -f https://raw.githubusercontent.com/projectcalico/calico/v3.29.4/manifests/tigera-operator.yaml

wget https://raw.githubusercontent.com/projectcalico/calico/v3.29.4/manifests/custom-resources.yaml


In [ ]:
vi custom-resources.yaml

In [ ]:
apiVersion: operator.tigera.io/v1
kind: Installation
spec:
  calicoNetwork:
    ipPools:
    - cidr: 10.244.0.0/16   # Match your pod CIDR from kubeadm init
      encapsulation: VXLAN   # or "IPIP" if better for your network
      natOutgoing: Enabled
    mtu: 1450               # Important for VXLAN
  # Add nodeAddressAutodetection if needed

In [ ]:
kubectl create -f custom-resources.yaml

---

---

use kube-vip as daemonset

In [ ]:
kube-vip manifest daemonset \
    --interface $INTERFACE \
    --inCluster \
    --services \
    --bgp \
    --localAS 64512 \
    --bgpRouterID $(hostname -I | awk '{print $1}') \
    --leaderElection > kube-vip-bgp-ds.yaml

In [ ]:
vim kube-vip-bgp-ds.yaml

In [ ]:
env:
        - name: bgp_local_as
          value: "64512"
        - name: bgp_router_id
          valueFrom:
            fieldRef:
              fieldPath: status.hostIP
        - name: vip_address
          value: "172.16.6.90"  
        - name: cp_enable
          value: "false"
        - name: svc_election
          value: "true"

        - name: bgp_peer_as
          value: "64512"  # <-change